# 2.2 多模态应用 - DocumentParser

构建支持PDF/图片/文本的多格式文档解析器, 为RAG系统提供文档摄入能力

## 项目定位

本实验的 DocumentPipeline 将被3.2和6.3项目使用

In [ ]:
import sys, os
sys.path.insert(0, "..")
from config import get_client, verify_config
from agent_project import *
from agent_project.tools import create_default_registry

client = get_client()
print(f"项目模块已加载 | LLM: {client.name} ({client.model})")


In [ ]:
# 文档解析器测试
from agent_project.pipeline import DocumentLoader, TextChunker, DocumentPipeline, Document

# 创建样例知识库文档
SAMPLE_DOCS = [
    {"source": "intro.md", "content": """# 智能知识库Agent介绍
智能知识库Agent是一个基于RAG和Agent技术的问答系统。它结合了向量检索和关键词检索,
通过混合检索策略提高召回率。同时集成了Agent推理能力, 可以调用外部工具获取信息。
核心技术栈: LangGraph + FAISS + BM25 + Qwen3.7。"""},
    {"source": "rag.md", "content": """# RAG技术详解
RAG(Retrieval-Augmented Generation)检索增强生成是2026年最主流的LLM应用架构。
工作流程: 用户提问 -> 文档检索 -> 上下文构建 -> LLM生成 -> 引用标注。
高级技术: 混合检索(BM25+向量)、RRF融合、Cross-Encoder重排序、Self-RAG。"""},
    {"source": "mcp.md", "content": """# MCP协议介绍
MCP(Model Context Protocol)是AI Agent的标准化协议。2025年12月捐赠Linux Foundation。
定义三种原语: Tool(操作工具)、Resource(只读数据)、Prompt(可复用模板)。
截至2026年6月, 已有10000+公开MCP服务器和9700万月SDK下载量。"""},
]

pipeline = DocumentPipeline(chunk_size=200, chunk_overlap=30)
pipeline.ingest_texts(SAMPLE_DOCS)

stats = pipeline.get_stats()
print(f"文档管道就绪: {stats['documents']}文档, {stats['total_chunks']}块")
print(f"均块大小: {stats['avg_chunk_size']:.0f}字符, 总字符: {stats['total_chars']}")


In [ ]:
# 展示文档分块结果
for doc in pipeline.documents:
    print(f"\n文档: {doc.metadata['source']} ({len(doc.content)}字符) -> {len(doc.chunks)}块")
    for i, chunk in enumerate(doc.chunks[:2]):
        print(f"  块[{i}]: {chunk[:80]}...")


In [ ]:
# 实际应用: 将分块结果传递给检索引擎
from agent_project.hybrid_search import HybridSearcher

searcher = HybridSearcher()
searcher.index(pipeline.all_chunks)

print(f"检索引擎已索引 {len(pipeline.all_chunks)} 个文本块")
print("就绪: DocumentPipeline -> HybridSearcher 管道连通!")
